In [73]:
import numpy as np
import pandas as pd
import mlflow
import onnx
import os
import onnxruntime as rt
from onnxmltools import convert_xgboost
from onnxmltools.utils import save_model
from onnxmltools.convert.common.data_types import FloatTensorType
import sys
sys.path.append('..')
from src.train import get_mlflow_client
import time
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

In [74]:
mlflow_client = get_mlflow_client("http://127.0.0.1:5000", "xgboost_training")

In [75]:
model = mlflow.xgboost.load_model("models:/xgboost_tsd_model@champion")

In [76]:
mlflow.end_run()

🏃 View run industrious-deer-882 at: http://127.0.0.1:5000/#/experiments/0/runs/e3ef8b009e8146eca0fc4ffb0acbb4e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


In [77]:
booster = model.get_booster()
booster.feature_names = None

In [78]:
onnx_model = convert_xgboost(
    model, 
    initial_types=[('input', FloatTensorType([None, model.n_features_in_]))]
)

In [79]:
models_dir = os.path.abspath("models")
os.makedirs(models_dir, exist_ok=True)
save_model(onnx_model, os.path.join(models_dir, "xgboost_tsd_model.onnx"))
print(f"Saved to: {os.path.join(models_dir, 'xgboost_tsd_model.onnx')}")

Saved to: C:\Users\ranas\OneDrive\Desktop\TSD\models\xgboost_tsd_model.onnx


In [80]:
test_data = pd.read_csv('data/test_x.csv')
test_data = test_data.astype(np.float32)

In [81]:
onnx_inputs = {input_name: test_data.values}
for _ in range(10):
    onnx_outputs = runtime.run(None, onnx_inputs)
start = time.perf_counter()
for _ in range(1000):
    onnx_outputs = runtime.run(None, onnx_inputs)
end = time.perf_counter()
elapsed_time_xgboost_onnx = end - start
print(f"ONNX Runtime inference time: {elapsed_time_xgboost_onnx:.4f} seconds")

ONNX Runtime inference time: 11.0546 seconds


In [82]:
for _ in range(10):
    outputs = model.predict(test_data)
start = time.perf_counter()
for _ in range(1000):
    outputs = model.predict(test_data)
end = time.perf_counter()
elapsed_time_xgboost = end - start
print(f"XGBoost inference time: {elapsed_time_xgboost:.4f} seconds")

XGBoost inference time: 7.1259 seconds


In [83]:
speedup = elapsed_time_xgboost / elapsed_time_xgboost_onnx
print(f"ONNX Runtime is {speedup:.1f}x faster than native XGBoost")

ONNX Runtime is 0.6x faster than native XGBoost


In [84]:
model_proto = onnx.load(str(Path(models_dir) / "xgboost_tsd_model.onnx"))

# Add missing ai.onnx opset
opset = model_proto.opset_import.add()
opset.domain = "ai.onnx"
opset.version = 17

onnx.save(model_proto, str(Path(models_dir) / "xgboost_tsd_model_fixed.onnx"))
print("Fixed model saved")

Fixed model saved


In [85]:
quantize_dynamic(
    model_input=str(Path(models_dir) / "xgboost_tsd_model_fixed.onnx"),
    model_output=str(Path(models_dir) / "xgboost_tsd_model_quantized.onnx"),
    weight_type=QuantType.QInt8
)

In [86]:
size_in_bytes_onnx = Path(models_dir) / "xgboost_tsd_model_fixed.onnx"
print(f"ONNX model size: {size_in_bytes_onnx.stat().st_size / (1024 * 1024):.2f} MB")

ONNX model size: 0.23 MB


In [87]:
size_in_bytes_onnx = Path(models_dir) / "xgboost_tsd_model_quantized.onnx"
print(f"ONNX model size: {size_in_bytes_onnx.stat().st_size / (1024 * 1024):.2f} MB")

ONNX model size: 0.23 MB


In [90]:

onnx_model = onnx.load("models/xgboost_tsd_model_fixed.onnx")
mlflow_client,experiment_id = get_mlflow_client("http://127.0.0.1:5000","xgboost_training")
with mlflow.start_run(experiment_id=experiment_id) as run:
    mlflow.onnx.log_model(onnx_model=onnx_model,artifact_path="model")
run_id = run.info.run_id
model_version = mlflow.register_model(f"runs:/{run_id}/model", "xgboost_tsd_model_fixed")
print(f"Model registered with version: {model_version.version}")
mlflow_client.set_registered_model_alias(name="xgboost_tsd_model_fixed", version=model_version.version, alias="champion")

2026/06/07 21:00:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'xgboost_tsd_model_fixed' already exists. Creating a new version of this model...
2026/06/07 21:00:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 0963ae0bde5d4cbea7cbd48753dc96b0 has no artifacts at artifact path 'model', registering model based on models:/m-9af0f888ebf84606a9cf63ad4c0e61b9 instead


🏃 View run gifted-goat-222 at: http://127.0.0.1:5000/#/experiments/2/runs/0963ae0bde5d4cbea7cbd48753dc96b0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/06/07 21:00:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_tsd_model_fixed, version 1
Created version '1' of model 'xgboost_tsd_model_fixed'.


Model registered with version: 1


In [91]:
v = mlflow_client.get_model_version_by_alias("xgboost_tsd_model_fixed", "champion")
print(v.version, v.source)

1 models:/m-9af0f888ebf84606a9cf63ad4c0e61b9


In [92]:
import mlflow
import os
mlflow.set_tracking_uri("http://localhost:5000")
path = mlflow.artifacts.download_artifacts("models:/xgboost_tsd_model_fixed@champion")
print("path:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.join(root, f))

path: C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\conda.yaml
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\MLmodel
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\model.onnx
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\python_env.yaml
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\registered_model_meta
C:\Users\ranas\AppData\Local\Temp\tmp07fq6b61\requirements.txt
